In [31]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_score, confusion_matrix, classification_report, roc_auc_score, roc_curve

cwd= os.getcwd()
path = os.path.join(cwd,'data')

In [32]:
test = pd.read_csv(os.path.join(path,'test.csv'))
train = pd.read_csv(os.path.join(path,'train.csv'))
gsub = pd.read_csv(os.path.join(path,'gender_submission.csv'))
print(train.shape, test.shape)
train.head()


train['Deck'] = train['Cabin'].str[0]
train['Age'] = train['Age'].replace(np.nan, train['Age'].median())

df = pd.concat([gsub, test], axis=1, ignore_index=True).drop(columns=[2])
df['Deck'] = df[11].str[0]
df.columns = train.columns.to_list()
df['Age'] = df['Age'].replace(np.nan, df['Age'].median())
df

(891, 12) (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Deck
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q,NaN
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S,NaN
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q,NaN
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S,NaN
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,1305,0,3,"Spector, Mr. Woolf",male,27.0,0,0,A.5. 3236,8.0500,NaN,S,NaN
414,1306,1,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C,C
415,1307,0,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S,NaN
416,1308,0,3,"Ware, Mr. Frederick",male,27.0,0,0,359309,8.0500,NaN,S,NaN


In [33]:
dataframe = pd.concat([train, df], axis=0, ignore_index=True)
print(dataframe.shape)
dataframe

(1309, 13)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Deck
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,NaN
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,NaN
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,C
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1304,1305,0,3,"Spector, Mr. Woolf",male,27.0,0,0,A.5. 3236,8.0500,NaN,S,NaN
1305,1306,1,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C,C
1306,1307,0,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S,NaN
1307,1308,0,3,"Ware, Mr. Frederick",male,27.0,0,0,359309,8.0500,NaN,S,NaN


In [34]:
le = LabelEncoder()
dataframe['Sex'] = le.fit_transform(dataframe['Sex'])
dataframe['Embarked'] = le.fit_transform(dataframe['Embarked'])
dataframe

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Deck
0,1,0,3,"Braund, Mr. Owen Harris",1,22.0,1,0,A/5 21171,7.2500,NaN,2,NaN
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",0,38.0,1,0,PC 17599,71.2833,C85,0,C
2,3,1,3,"Heikkinen, Miss. Laina",0,26.0,0,0,STON/O2. 3101282,7.9250,NaN,2,NaN
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",0,35.0,1,0,113803,53.1000,C123,2,C
4,5,0,3,"Allen, Mr. William Henry",1,35.0,0,0,373450,8.0500,NaN,2,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1304,1305,0,3,"Spector, Mr. Woolf",1,27.0,0,0,A.5. 3236,8.0500,NaN,2,NaN
1305,1306,1,1,"Oliva y Ocana, Dona. Fermina",0,39.0,0,0,PC 17758,108.9000,C105,0,C
1306,1307,0,3,"Saether, Mr. Simon Sivertsen",1,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,2,NaN
1307,1308,0,3,"Ware, Mr. Frederick",1,27.0,0,0,359309,8.0500,NaN,2,NaN


In [35]:
dataframe = dataframe.drop(
    columns=['PassengerId', 'Name', 'Ticket', 'Cabin', 'Deck'])

x = dataframe.drop(columns=['Survived'])
y = dataframe['Survived']

x.shape, y.shape

((1309, 7), (1309,))

In [36]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42, stratify=y)

for name, arr in zip(['x_train', 'x_test', 'y_train', 'y_test'], [x_train, x_test, y_train, y_test]):
    print(f'{name}: {arr.shape}')

x_test['Fare'] = x_test['Fare'].fillna(x_test['Fare'].mean())

x_train: (981, 7)
x_test: (328, 7)
y_train: (981,)
y_test: (328,)


In [ ]:
# RandomForestClassifier
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier())
])

param_grid = {
    "clf__n_estimators": [100, 200, 300],
    "clf__max_depth": [5, 10, 20],
    "clf__min_samples_split": [2, 5, 10],
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='precision', n_jobs=-1)
grid.fit(x_train, y_train)

print("train accuracy = {:.3%}".format(grid.score(x_train, y_train)))
print("test accuracy = {:.3%}".format(grid.score(x_test, y_test)))
print("Best params:", grid.best_params_)

best_model = grid.best_estimator_
fi = pd.Series(best_model.named_steps['clf'].feature_importances_, index=x_train.columns).sort_values(ascending=False)
print("Feature Importances:", dict(fi))

train accuracy = 93.567%
test accuracy = 88.000%
Best params: {'clf__max_depth': 10, 'clf__min_samples_split': 5, 'clf__n_estimators': 100}
Feature Importances: {'Sex': np.float64(0.5259927999738585), 'Fare': np.float64(0.16654213392515013), 'Age': np.float64(0.13850807093111792), 'Pclass': np.float64(0.06865575665183714), 'Parch': np.float64(0.040654132596036126), 'SibSp': np.float64(0.03744078841363119), 'Embarked': np.float64(0.022206317508369107)}


In [ ]:
# GradientBoostingClassifier
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", GradientBoostingClassifier())
])

param_grid = {
    "clf__n_estimators": [100, 200, 300],
    "clf__learning_rate": [0.01, 0.1, 0.2],
    "clf__max_depth": [3, 5, 7]
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid.fit(x_train, y_train)

print("train accuracy = {:.3%}".format(grid.score(x_train, y_train)))
print("test accuracy = {:.3%}".format(grid.score(x_test, y_test)))
print("Best params:", grid.best_params_)

best_model = grid.best_estimator_
fi = pd.Series(best_model.named_steps['clf'].feature_importances_, index=x_train.columns).sort_values(ascending=False)
print("Feature Importances:", dict(fi))

train accuracy = 96.207%
test accuracy = 87.520%
Best params: {'clf__learning_rate': 0.01, 'clf__max_depth': 5, 'clf__n_estimators': 200}
Feature Importances: {'Sex': np.float64(0.6759380887895575), 'Fare': np.float64(0.1383449550728512), 'Age': np.float64(0.08580470288053474), 'Pclass': np.float64(0.057484263146260074), 'SibSp': np.float64(0.02467335783349475), 'Parch': np.float64(0.013697956331507768), 'Embarked': np.float64(0.0040566759457939475)}


In [ ]:
# XGBClassifier
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", XGBClassifier())
])

param_grid = {
    'clf__tree_method': ['auto'],
    'clf__learning_rate': [0.1, 0.2, 0.3],
    'clf__max_depth': [3,4,5,6]
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid.fit(x_train, y_train)

print("train accuracy = {:.3%}".format(grid.score(x_train, y_train)))
print("test accuracy = {:.3%}".format(grid.score(x_test, y_test)))
print("Best params:", grid.best_params_)

best_model = grid.best_estimator_
fi = pd.Series(best_model.named_steps['clf'].feature_importances_, index=x_train.columns).sort_values(ascending=False)
print("Feature Importances:", dict(fi))

train accuracy = 97.480%
test accuracy = 88.067%
Best params: {'clf__learning_rate': 0.1, 'clf__max_depth': 5, 'clf__tree_method': 'auto'}
Feature Importances: {'Sex': np.float32(0.79603), 'Pclass': np.float32(0.09452665), 'Fare': np.float32(0.02848077), 'Age': np.float32(0.0243901), 'SibSp': np.float32(0.024323707), 'Parch': np.float32(0.018119674), 'Embarked': np.float32(0.0141292205)}
